In [1]:
import os
print(os.getcwd())

import sys
sys.path.append('../scripts')
from importlib import import_module
ca_module = import_module('08_context_assembler')

assembler = ca_module.ContextAssembler()

# 拿几个真实chunk测token估算准不准
test_texts = [
    "Metformin is widely used as first-line therapy for type 2 diabetes.",
    "二甲双胍是2型糖尿病一线用药，通过抑制肝糖异生发挥降糖作用。",
    "Metformin (二甲双胍) reduces cardiovascular risk in diabetic patients through AMPK activation pathways."
]

for t in test_texts:
    n_tokens = assembler.estimate_tokens(t)
    print(f"[{n_tokens} tokens] {t[:50]}...")


/Users/siqinling/medrag_project/notebooks


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
[16 tokens] Metformin is widely used as first-line therapy for...
[22 tokens] 二甲双胍是2型糖尿病一线用药，通过抑制肝糖异生发挥降糖作用。...
[21 tokens] Metformin (二甲双胍) reduces cardiovascular risk in di...


In [2]:
import sys
sys.path.append('../scripts')
from importlib import import_module

pipeline_module = import_module('07_retrieval_pipeline')
pipeline = pipeline_module.MedRAGPipeline(
    chunks_path="../data/processed/chunks.parquet",
    chroma_db_path="../data/processed/chroma_db"
)


result = pipeline.run("What is the effect of metformin on cardiovascular disease?")


print(type(result))
print(result.keys() if isinstance(result, dict) else "不是dict")


/opt/anaconda3/envs/medrag/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


初始化 MultiPathRetriever...
加载chunk数据...
连接ChromaDB...
加载embedding模型...


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 1592.59it/s]
Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/y5/4900bss12yg8nhh_4m3wz5q00000gn/T/jieba.cache


构建BM25索引...


Loading model cost 0.263 seconds.
Prefix dict has been built successfully.


初始化 Reranker...
加载reranker模型...


Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 9979.11it/s]


<class 'list'>
不是dict


In [3]:
print(len(result))
print(type(result[0]))
print(result[0])

10
<class 'dict'>
{'chunk_id': 'PMC2946277_chunk0', 'text': 'Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention. Materials and methods All patients aged 30 years or older receiving glucose-lowering drugs (GLDs) and admitted with myocardial infarction (MI) not treated with emergent percutaneous coronary intervention in Denmark during 1997-2006 were identified by individual-level lin

In [4]:
converted = assembler._convert_to_chunks(result)
print(len(converted))
print(converted[0])


10
DocumentChunk(text='Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention. Materials and methods All patients aged 30 years or older receiving glucose-lowering drugs (GLDs) and admitted with myocardial infarction (MI) not treated with emergent percutaneous coronary intervention in Denmark during 1997-2006 were identified by individual-level linkage of nationwide registries of hospi

In [5]:
deduped = assembler._deduplicate(converted)
print(f"去重前: {len(converted)}, 去重后: {len(deduped)}")

# 顺手看看有没有明显相似的chunk被识别出来
for i in range(len(converted)):
    for j in range(i+1, len(converted)):
        sim = assembler._jaccard_similarity(converted[i].text, converted[j].text)
        if sim > 0.3:
            print(f"chunk {i} vs {j}: 相似度 {sim:.2f}")


去重前: 10, 去重后: 10


In [6]:
test_chunks = [
    ca_module.DocumentChunk(text="A1", metadata={}, relevance_score=0.9, source="PMC001", chunk_id="PMC001_chunk0"),
    ca_module.DocumentChunk(text="A2", metadata={}, relevance_score=0.85, source="PMC001", chunk_id="PMC001_chunk1"),
    ca_module.DocumentChunk(text="A3", metadata={}, relevance_score=0.8, source="PMC001", chunk_id="PMC001_chunk2"),
    ca_module.DocumentChunk(text="B1", metadata={}, relevance_score=0.75, source="PMC002", chunk_id="PMC002_chunk0"),
    ca_module.DocumentChunk(text="C1", metadata={}, relevance_score=0.7, source="PMC003", chunk_id="PMC003_chunk0"),
]

ranked = assembler._diversify_and_rank(test_chunks)
for c in ranked:
    print(f"{c.chunk_id}: {c.relevance_score}")


PMC001_chunk0: 0.9
PMC002_chunk0: 0.75
PMC003_chunk0: 0.7
PMC001_chunk1: 0.85
PMC001_chunk2: 0.8


In [7]:
final_result = assembler.assemble_context(result)

print(final_result["metadata"])
print("---")
print(final_result["context_text"][:1000])


{'total_chunks_retrieved': 10, 'unique_chunks_after_dedup': 10, 'chunks_selected': 8, 'estimated_tokens': 2985, 'chunk_sources': {'unique_sources': 8, 'source_distribution': {'PMC2946277': 1, 'PMC2546413': 1, 'PMC2566605': 1, 'PMC2991324': 1, 'PMC1974811': 1, 'PMC2664796': 1, 'PMC2940872': 1, 'PMC2797799': 1}}}
---
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated

In [8]:
small_assembler = ca_module.ContextAssembler(max_context_tokens=200)
small_result = small_assembler.assemble_context(result)
print(small_result["metadata"])
print(small_result["context_text"])


[ContextAssembler] tokenizer 加载成功: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
{'total_chunks_retrieved': 10, 'unique_chunks_after_dedup': 10, 'chunks_selected': 1, 'estimated_tokens': 144, 'chunk_sources': {'unique_sources': 1, 'source_distribution': {'PMC2946277': 1}}}
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emergent percutaneous coronary intervention - a retrospective nationwide cohort study. Background The optimum oral pharmacological treatment of diabetes mellitus to reduce cardiovascular disease and mortality following myocardial infarction has not been established. We therefore set out to investigate the association between individual oral glucose-lowering drugs and cardiovascular outcomes following myocardial infarction in patients with diabetes mellitus not treated with emergent percutaneous coronary intervention

In [9]:
import sys
sys.path.append('../scripts')
from importlib import import_module

pt_module = import_module('09_prompt_templates')

stage = pt_module.EVIDENCE_EVALUATOR
filled_user_prompt = stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500]
)
print(stage.system_prompt)
print("---")
print(filled_user_prompt)


You are a medical evidence evaluator. Your task is to critically assess a set of retrieved literature excerpts in relation to a clinical or biomedical question.

For each excerpt, evaluate:
1. Relevance: Does it directly address the question, or only tangentially related?
2. Evidence strength: What type of study is it (e.g. RCT, cohort study, case report, review)? Larger, controlled, and more recent studies generally carry more weight.
3. Consistency: Does it agree or conflict with other excerpts provided?

Do not answer the question yet. Only produce a structured evaluation of the evidence. Be concise and avoid restating the full text of each excerpt — reference them by source ID.
---
Question: What is the effect of metformin on cardiovascular disease?

Retrieved evidence:
[来源: PMC2946277 | 期刊: Cardiovascular Diabetology | 发表: 2010-9-16]
Effects of oral glucose-lowering drugs on long term outcomes in patients with diabetes mellitus following myocardial infarction not treated with emer

In [10]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

ag_stage = pt_module.ANSWER_GENERATOR
filled = ag_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500],
    evidence_evaluation="[占位] Source PMC2946277: relevance high, cohort study, moderate evidence strength."
)
print(ag_stage.system_prompt)
print("---")
print(filled)


You are a medical literature assistant. Your task is to draft an evidence-based answer to a clinical or biomedical question, using only the retrieved literature excerpts and the evidence evaluation provided.

Rules:
1. Base your answer strictly on the provided evidence. Do not introduce facts, mechanisms, or statistics that are not present in the excerpts.
2. Prioritize evidence rated as high relevance and strong evidence quality in the evaluation. Give less weight to low-relevance or weak-quality sources, and mention if evidence is limited or mixed.
3. Cite each claim with its source ID (e.g. [PMC2946277]) immediately after the statement it supports.
4. If the evidence is insufficient or conflicting on some aspect of the question, say so explicitly rather than filling the gap with assumptions.
5. Write in clear, precise scientific English suitable for a clinical audience.
---
Question: What is the effect of metformin on cardiovascular disease?

Retrieved evidence:
[来源: PMC2946277 | 期刊

In [11]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

cr_stage = pt_module.CRITICAL_REVIEWER
filled = cr_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    context=final_result["context_text"][:500],
    draft_answer="[占位] Metformin reduces cardiovascular mortality in diabetic patients [PMC2946277]."
)
print(cr_stage.system_prompt)
print("---")
print(filled)


You are a critical reviewer specializing in evidence-based medicine. Your task is to fact-check a draft answer against the original retrieved evidence, and identify any issues.

Check specifically for:
1. Hallucination: Does the draft state any fact, number, or mechanism that is NOT actually present in the retrieved evidence?
2. Citation accuracy: Does each cited source ID actually support the claim it's attached to?
3. Overreach: Does the draft state a causal conclusion when the underlying evidence only supports a correlational or associative finding (e.g. observational/cohort studies)?
4. Missed uncertainty: Are there points where evidence is weak, limited, or conflicting, but the draft states them with unwarranted confidence?

Do not rewrite the answer yourself. Only produce a list of issues found, each with a brief explanation and, where applicable, a suggested correction. If no issues are found for a category, state that explicitly.
---
Question: What is the effect of metformin on

In [12]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

fa_stage = pt_module.FINAL_ASSEMBLER
filled = fa_stage.user_prompt_template.format(
    query="What is the effect of metformin on cardiovascular disease?",
    draft_answer="[占位] Metformin reduces cardiovascular mortality in diabetic patients [PMC2946277].",
    review_issues="[占位] Overreach: the cited study is a cohort study, so 'reduces' should be softened to 'associated with lower' unless a causal design is confirmed."
)
print(fa_stage.system_prompt)
print("---")
print(filled)


You are a medical literature assistant preparing a final answer for a Chinese-speaking user. You will be given a draft answer (in English) and a list of issues found during critical review.

Your task:
1. Revise the draft to address every issue raised in the review — remove any unsupported claims, soften overreaching causal language into correlational language where appropriate, and add explicit notes of uncertainty where the review flagged weak or conflicting evidence.
2. Keep all source citations (e.g. [PMC2946277]) attached to their corresponding claims.
3. Translate the final, corrected answer into clear, natural Chinese suitable for a medical student or researcher. Do not simply translate the flawed draft — translate the corrected version.
4. If the evidence is genuinely insufficient to answer part of the question, state this clearly in Chinese rather than omitting it silently.

Output only the final Chinese answer. Do not include your revision process or the English draft.
---
Qu

In [13]:
if '09_prompt_templates' in sys.modules:
    del sys.modules['09_prompt_templates']
pt_module = import_module('09_prompt_templates')

for stage_key, stage in pt_module.PROMPT_STAGES.items():
    print(f"{stage_key} -> {stage.name} (temperature={stage.temperature}, max_tokens={stage.max_tokens})")

evidence_evaluator -> 证据评估器 (temperature=0.2, max_tokens=800)
answer_generator -> 答案生成器 (temperature=0.3, max_tokens=1000)
critical_reviewer -> 批判性审查器 (temperature=0.2, max_tokens=800)
final_assembler -> 最终组装器 (temperature=0.3, max_tokens=1200)
